# 第08回 深層学習2：値とコードを一段ずつ読む

入力を再構成する、時間を通じて記憶する、作る側と見分ける側を交互に更新する、の三つを比べます。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** 全25出力のMSE約0.316058→0.004639、手動ゲートc=1.1、D→Gの更新で固定側が不変。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
import copy


In [ ]:
import json


In [ ]:
import math


In [ ]:
SEED = 65535


In [ ]:
DATA = [[float(i == j) for j in range(5)] for i in range(5)]


In [ ]:
REAL = [1.0, 2.0, 3.0]


In [ ]:
NOISE = [-1.0, 0.0, 1.0]


In [ ]:
GAN_INITIAL = {"a": 0.5, "b": -1.0, "w": 1.0, "d": 0.0}


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 5値から3値、再び5値へ

**今いる段階：** 入力→積と和→中間→再構成→誤差

**予想：** どの値が再構成の正解か

**操作：** example_positionを0〜4へ変更

**結果の目印：** 入力自身が正解。初期と1000回後の5個の出力が図に並ぶ

**戻す：** example_position=0で再実行

**説明する：** 1入力のMSEと全入力のMSEの分母の違い


### 計算の定義：`initial_autoencoder`

JSでも再現できる32bit LCG。教材用で暗号用途ではない。


In [ ]:
def initial_autoencoder(seed=SEED):
    """JSでも再現できる32bit LCG。教材用で暗号用途ではない。"""
    state = seed
    def value():
        nonlocal state
        state = (1664525 * state + 1013904223) % (2 ** 32)
        return 2 * state / (2 ** 32) - 1
    return {"enc": [[value() for _ in range(6)] for _ in range(3)],
            "dec": [[value() for _ in range(4)] for _ in range(5)]}


### 計算の定義：`sigmoid`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def sigmoid(x):
    if x >= 0:
        return 1 / (1 + math.exp(-x))
    e = math.exp(x)
    return e / (1 + e)


### 計算の定義：`dense_layer`

入力×重みを順に並べ、バイアスを足してsigmoidへ渡す。


In [ ]:
def dense_layer(values, rows, record=None):
    """入力×重みを順に並べ、バイアスを足してsigmoidへ渡す。"""
    output = []
    for unit, weights in enumerate(rows):
        products = []
        for index, value in enumerate(values):
            products.append(weights[index] * value)
        bias = weights[-1]
        total = sum(products) + bias
        activated = sigmoid(total)
        output.append(activated)
        if record is not None:
            record({'unit': unit, 'values': list(values), 'weights': list(weights[:-1]),
                    'products': products, 'bias': bias, 'total': total, 'output': activated})
    return output


### 計算の定義：`reconstruct`

5値→3値→5値。目標は入力自身で、誤差は5個の二乗差の平均。


In [ ]:
def reconstruct(x, weights):
    """5値→3値→5値。目標は入力自身で、誤差は5個の二乗差の平均。"""
    if len(x) != 5:
        raise ValueError("入力は5個の数値です")
    hidden = dense_layer(x, weights['enc'])
    output = dense_layer(hidden, weights['dec'])
    errors = []
    for index, value in enumerate(output):
        errors.append((value - x[index]) ** 2)
    return {'hidden': hidden, 'output': output, 'mse': sum(errors) / 5}


### 計算の定義：`ae_gradient`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def ae_gradient(weights, data=DATA):
    grad = {k: [[0.0 for _ in row] for row in rows] for k, rows in weights.items()}
    for x in data:
        r = reconstruct(x, weights)
        h, y = r["hidden"], r["output"]
        dy = [2 * (y[k] - x[k]) * y[k] * (1 - y[k]) / (5 * len(data))
              for k in range(5)]
        # 出力側も中間層側も、更新前の同じ重みから勾配を計算する。
        dh = [sum(dy[k] * weights["dec"][k][j] for k in range(5)) * h[j] * (1 - h[j])
              for j in range(3)]
        for k in range(5):
            for j in range(4):
                grad["dec"][k][j] += dy[k] * (h[j] if j < 3 else 1)
        for j in range(3):
            for i in range(6):
                grad["enc"][j][i] += dh[j] * (x[i] if i < 5 else 1)
    return grad


### 計算の定義：`ae_loss`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def ae_loss(weights, data=DATA):
    return sum(reconstruct(x, weights)["mse"] for x in data) / len(data)


### 計算の定義：`train_autoencoder`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def train_autoencoder(steps=1000, rate=4.0, data=DATA, initial=None):
    if not isinstance(steps, int) or not 0 <= steps <= 4000 or not 0 <= rate <= 4:
        raise ValueError("学習回数は0〜4000の整数、学習率は0〜4です")
    weights = copy.deepcopy(initial if initial is not None else initial_autoencoder())
    history = [{"step": 0, "mse": ae_loss(weights, data)}]
    for step in range(1, steps + 1):
        grad = ae_gradient(weights, data)
        for key, rows in weights.items():
            for i, row in enumerate(rows):
                for j in range(len(row)):
                    row[j] -= rate * grad[key][i][j]
        if step % 100 == 0 or step == steps:
            history.append({"step": step, "mse": ae_loss(weights, data)})
    return {"weights": weights, "history": history,
            "reconstructions": [reconstruct(x, weights) for x in data]}


### 計算の定義：`lesson_autoencoder`

入力の1の位置を選び、最初の中間ユニットと再構成を追う。


In [ ]:
def lesson_autoencoder(position=0):
    """入力の1の位置を選び、最初の中間ユニットと再構成を追う。"""
    x = DATA[position][:]
    weights = initial_autoencoder()
    records = []
    hidden = dense_layer(x, weights['enc'], records.append)
    first = records[0]
    before = reconstruct(x, weights)
    fitted = train_autoencoder()
    after = reconstruct(x, fitted['weights'])
    rows = [(j,x[j],first['weights'][j],first['products'][j]) for j in range(5)]
    errors = [(j,x[j],before['output'][j],(before['output'][j]-x[j])**2,after['output'][j]) for j in range(5)]
    return {'title':'自己符号化器：5→3→5', 'question':'5値を3値にまとめても元の1の位置へ戻せるか',
            'input':x,'columns':['入力番号','入力','中間0への重み','積'],'rows':rows,
            'tables':[('中間0：積の合計→バイアス→sigmoid',['積の合計','バイアス','合計','出力'],[(sum(first['products']),first['bias'],first['total'],first['output'])]),
                      ('5個の二乗誤差を平均する',['位置','正解=入力','初期再構成','二乗差','1000回後'],errors)],
            'matrices':[('入力5値',[x]),('中間3値',[hidden]),('初期再構成5値',[before['output']]),('学習後5値',[after['output']])],
            'plots':[('100回ごとの実測MSE／横軸=更新回数',[(r['step'],r['mse']) for r in fitted['history']])],
            'result':f"この入力のMSE={before['mse']}; 全5入力の平均MSE={fitted['history'][0]['mse']}",
            'comparison':f"同じ入力で重みを1000回学習→MSE={after['mse']}",
            'explanation':'1入力は5出力の平均、学習全体は5入力・25出力の平均。入力を変えることと学習を分ける。',
            'reset':'position=0で再実行。毎回、同じ初期重みのコピーから学習する。'}


In [ ]:
example_position = 0
program_observation_autoencoder = lesson_autoencoder(example_position)
learning_show(program_observation_autoencoder, reset='example_position=0で再実行')


## 2. 時刻ごとに記憶を更新する

**今いる段階：** 前のc→残す量＋書く量→新しいc→h

**予想：** 0入力でも記憶はすぐ消えるか

**操作：** example_forgetを0.8→0へ

**結果の目印：** 0.8なら1→0.8→0.64→0.512、0なら直後から0

**戻す：** example_forget=0.8

**説明する：** 時刻更新と重み学習は何が違うか


### 計算の定義：`gate_step`

ゲートを手動指定したセル計算。ゲートの学習実験ではない。


In [ ]:
def gate_step(previous=1.0, forget=0.8, input_gate=0.5, candidate=0.6, output_gate=0.75):
    """ゲートを手動指定したセル計算。ゲートの学習実験ではない。"""
    if not all(0 <= v <= 1 for v in (forget, input_gate, output_gate)) or not -1 <= candidate <= 1:
        raise ValueError("ゲートは0〜1、候補は-1〜1です")
    kept = forget * previous
    written = input_gate * candidate
    cell = kept + written
    return {"kept": kept, "written": written, "cell": cell,
            "hidden": output_gate * math.tanh(cell)}


### 計算の定義：`memory_sequence`

入力・出力ゲートを1とした理想化例。候補値は生の入力とは区別する。


In [ ]:
def memory_sequence(candidates=(1.0, 0.0, 0.0, 0.0), forget=0.8):
    """入力・出力ゲートを1とした理想化例。候補値は生の入力とは区別する。"""
    cell = 0.0
    result = []
    for candidate in candidates:
        step = gate_step(cell, forget, 1, candidate, 1)
        cell = step["cell"]
        result.append(step)
    return result


### 計算の定義：`fixed_lstm_step`

固定した重みからゲートを計算する1次元LSTM（未学習）。


In [ ]:
def fixed_lstm_step(x, previous_cell=0.0, previous_hidden=0.0):
    """固定した重みからゲートを計算する1次元LSTM（未学習）。"""
    f = sigmoid(0.5 * x + 0.2 * previous_hidden + 1)
    i = sigmoid(x - 0.1 * previous_hidden)
    g = math.tanh(x + 0.3 * previous_hidden)
    o = sigmoid(0.4 * x + 0.1 * previous_hidden)
    return dict(gate_step(previous_cell, f, i, g, o), forget=f, input_gate=i,
                candidate=g, output_gate=o)


### 計算の定義：`lesson_lstm`

手動ゲートの記憶と、固定重みLSTMの時刻別計算を区別する。


In [ ]:
def lesson_lstm(forget=0.8):
    """手動ゲートの記憶と、固定重みLSTMの時刻別計算を区別する。"""
    candidates = [1.,0.,0.,0.]
    states = memory_sequence(candidates, forget)
    previous, rows = 0., []
    for time, (candidate, state) in enumerate(zip(candidates, states)):
        rows.append((time,previous,forget,state['kept'],candidate,state['written'],state['cell'],state['hidden']))
        previous = state['cell']
    fixed, cell, hidden = [], 0., 0.
    for time, x in enumerate([1.,0.,0.,0.]):
        result = fixed_lstm_step(x,cell,hidden)
        fixed.append((time,x,cell,result['forget'],result['input_gate'],result['candidate'],result['cell'],result['hidden']))
        cell, hidden = result['cell'], result['hidden']
    return {'title':'LSTM：前の記憶を残し、新しい情報を書く','question':'入力後に0が続くと記憶はどうなるか',
            'input':'候補[1,0,0,0]。手動例は入力・出力ゲート=1。固定重み例は生入力から各ゲートを計算。',
            'columns':['時刻','前のc','忘却f','残す量','候補g','書く量','新しいc','h'],'rows':rows,
            'tables':[('別のモデル：固定重みの1次元LSTM',['時刻','入力x','前のc','f','i','g','c','h'],fixed)],
            'result':f"手動例の最後の記憶={states[-1]['cell']}",
            'comparison':f"忘却を0にすると最後の記憶={memory_sequence(candidates,0)[-1]['cell']}",
            'explanation':'c=f×前のc+i×g。h=o×tanh(c)。時刻を進めるだけでは重みを学習していない。',
            'reset':'forget=0.8で再実行。時刻0のcとhは0から始まる。'}


In [ ]:
example_forget = 0.8
program_observation_lstm = lesson_lstm(example_forget)
learning_show(program_observation_lstm, reset='example_forget=0.8')


## 3. Dを更新してからGを更新する

**今いる段階：** 入力→Dの勾配と更新→Gの勾配と更新

**予想：** どの2個の重みを固定するか

**操作：** example_rateを0.1→0へ

**結果の目印：** D更新はw,dのみ、G更新はa,bのみ変わる

**戻す：** example_rate=0.1

**説明する：** G更新でどの時点のDを使うか


### 計算の定義：`generate`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def generate(z, params):
    return params["a"] * z + params["b"]


### 計算の定義：`discriminate`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def discriminate(x, params):
    return sigmoid(params["w"] * x + params["d"])


### 計算の定義：`gan_gradient`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def gan_gradient(params, target):
    if target == "d":
        grad = {"w": 0.0, "d": 0.0}
        for xs, label in ((REAL, 1), ([generate(z, params) for z in NOISE], 0)):
            for x in xs:
                delta = (discriminate(x, params) - label) / len(xs)
                grad["w"] += delta * x
                grad["d"] += delta
        return grad
    if target == "g":
        grad = {"a": 0.0, "b": 0.0}
        for z in NOISE:
            delta = (discriminate(generate(z, params), params) - 1) * params["w"] / len(NOISE)
            grad["a"] += delta * z
            grad["b"] += delta
        return grad
    raise ValueError("更新対象はdまたはgです")


### 計算の定義：`gan_step`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def gan_step(params, target, rate=0.1):
    if not 0 <= rate <= 1:
        raise ValueError("学習率は0〜1です")
    updated = dict(params)
    for key, value in gan_gradient(params, target).items():
        updated[key] -= rate * value
    return updated


### 計算の定義：`softplus`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def softplus(x):
    return max(x, 0) + math.log1p(math.exp(-abs(x)))


### 計算の定義：`gan_losses`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def gan_losses(params):
    real_logits = [params["w"] * x + params["d"] for x in REAL]
    fake_logits = [params["w"] * generate(z, params) + params["d"] for z in NOISE]
    return {"d_loss": sum(softplus(-s) for s in real_logits) / len(REAL)
            + sum(softplus(s) for s in fake_logits) / len(NOISE),
            "g_loss": sum(softplus(-s) for s in fake_logits) / len(NOISE)}


### 計算の定義：`lesson_gan`

Dだけ更新→更新後Dを固定してGだけ更新、という1往復。


In [ ]:
def lesson_gan(rate=0.1):
    """Dだけ更新→更新後Dを固定してGだけ更新、という1往復。"""
    initial = dict(GAN_INITIAL)
    d_gradient = gan_gradient(initial,'d')
    after_d = gan_step(initial,'d',rate)
    g_gradient = gan_gradient(after_d,'g')
    after_g = gan_step(after_d,'g',rate)
    rows = [(key,initial[key],d_gradient.get(key,'固定'),after_d[key],g_gradient.get(key,'固定'),after_g[key]) for key in ('a','b','w','d')]
    generated = [(z,generate(z,initial),generate(z,after_d),generate(z,after_g)) for z in NOISE]
    return {'title':'GAN：更新する側と固定する側','question':'Dを更新した直後、Gが作る値は変わるか',
            'input':f'実データ{REAL}、ノイズ{NOISE}、学習率{rate}',
            'columns':['重み','初期','D更新の勾配','D更新後','G更新の勾配','G更新後'],'rows':rows,
            'tables':[('G(z)=a×z+b。D更新中はa,bを固定',['z','初期G(z)','D更新後G(z)','G更新後G(z)'],generated)],
            'result':f'1往復後の損失 {gan_losses(after_g)}',
            'comparison':f"学習率0なら両更新後も初期値と同じ：{gan_step(gan_step(initial,'d',0),'g',0)==initial}",
            'explanation':'a,bはG、w,dはD。D更新中はGを固定し、その後のG更新では更新後のDを固定する。',
            'reset':'rate=0.1で再実行。既存の学習結果を引き継がずGAN_INITIALのコピーから始める。'}


In [ ]:
example_rate = 0.1
program_observation_gan = lesson_gan(example_rate)
learning_show(program_observation_gan, reset='example_rate=0.1')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`rounded`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def rounded(value):
    if isinstance(value, float):
        return round(value, 6)
    if isinstance(value, list):
        return [rounded(v) for v in value]
    if isinstance(value, dict):
        return {k: rounded(v) for k, v in value.items()}
    return value


### 計算の定義：`demo`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def demo():
    trained = train_autoencoder()
    after_d = gan_step(GAN_INITIAL, "d")
    after_g = gan_step(after_d, "g")
    return rounded({
        "autoencoder": {"before": ae_loss(initial_autoencoder()),
                        "after": trained["history"][-1]["mse"],
                        "first": trained["reconstructions"][0]},
        "gates": gate_step(), "sequence": memory_sequence(),
        "fixed_lstm": fixed_lstm_step(1),
        "gan": {"initial": dict(GAN_INITIAL), "after_d": after_d, "after_g": after_g,
                "generated": [generate(z, after_g) for z in NOISE], "losses": gan_losses(after_g)}})


## 4. 基準結果

まず下のセルを実行する前に、LSTMのセル状態を手計算する。実行後は自己符号化器MSE 0.316058→0.004639、gatesのc=1.1、h=0.600374を確認する。

GANのD更新後はa,bが不変、G更新後はw,dが不変であることを表す辞書を読む。値は表示用に6桁へ丸める。


In [ ]:
baseline_result = demo()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))


## 変更実験1：入力

予想：1の位置を変えると、中間表現と再構成はどう変わるか。固定初期重みで入力だけを比較する。リストのコピーを使うので元のデータ5行は変わらない。


In [ ]:
trial_x = DATA[1][:]
trial_reconstruction = reconstruct(trial_x, initial_autoencoder())
print("入力:", trial_x, "再構成:", rounded(trial_reconstruction))


## 変更実験2：学習回数

予想：同じ初期値から2,000回学習したらMSEはどうなるか。基準の1,000回とは別に実行する。想定結果は約0.001924。回数以外の条件は同じか確認する。


In [ ]:
trial_fit = train_autoencoder(2000)
print("2000回のMSE:", round(trial_fit["history"][-1]["mse"], 6))


## 変更実験3：出力ゲート

予想：o=0ならcも0になるか。想定結果はc=1.1、h=0。出力の抑制と忘却の違いを説明する。


In [ ]:
trial_gate = gate_step(output_gate=0)
print(rounded(trial_gate))


## 変更実験4：順序

予想：[1,0,0,0]を逆順にしたら最後の状態はどうなるか。コピーを逆順にし、元のリストを残す。最後のcは1となる。書き込んだ後に何回fを掛けたかを説明する。


In [ ]:
baseline_candidates = [1.0, 0.0, 0.0, 0.0]
trial_candidates = baseline_candidates[::-1]
trial_sequence = memory_sequence(trial_candidates)
print("元:", baseline_candidates, "変更後:", trial_candidates)
print(rounded(trial_sequence))


## 変更実験5：生成器のバイアス

予想：bを−1から0へ変えると、z=0の生成値と判定はどうなるか。想定はG(0)=0、D(G(0))=0.5。手動変更と学習を区別する。


In [ ]:
trial_params = dict(GAN_INITIAL)
trial_params["b"] = 0
trial_generated = generate(0, trial_params)
trial_decision = discriminate(trial_generated, trial_params)
print("生成値:", trial_generated, "判定:", trial_decision)


## 変更実験6：学習率を0へ

予想：GANで学習率が0ならどのパラメーターが変わるか。コピーからD→Gを更新する。全て初期値のままであることを確認する。


In [ ]:
trial_zero = gan_step(dict(GAN_INITIAL), "d", rate=0)
trial_zero = gan_step(trial_zero, "g", rate=0)
print("初期:", GAN_INITIAL, "変更後:", trial_zero)


## 結果を説明する・元に戻す

各実験で「変えた値」「固定した条件」「実際の出力」「その理由」を記録する。基準結果は `baseline_result` に残してある。

NameErrorなら再起動して全セル実行。予想と違うときは一つずつ条件を確認する。編集したコードを戻すには元のZIPから新しいフォルダーへ展開する。カーネルの再起動だけではソースの編集は戻らない。

Python版はvenv有効化後に `python deep_learning2.py --demo`。uvを使う場合は `uv run --no-project python deep_learning2.py --demo`。

## 参考資料

- [Deep Learning 第14章](https://www.deeplearningbook.org/contents/autoencoders.html)
- [PyTorch LSTMの式](https://docs.pytorch.org/docs/2.14/generated/torch.nn.LSTM.html)
- [Generative Adversarial Nets](https://papers.nips.cc/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html)

外部の参考資料の閲覧だけはネット接続が必要。今回の数値例は授業用の新作で、原論文の実験の再現ではない。


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python deep_learning2.py --explain all`で同じ計算表を表示します。各テーマは`--explain autoencoder`のように指定できます。従来のコマンドと固定期待値も維持しています。
